# 00 Dataset Audit
Goal:
- inspect the structure of the dataset files,
- assign meaningful column names,
- verify `txId` consistency across files,
- check missing values, duplicates, and basic value ranges,
- prepare key decisions for the next analysis notebooks.

## Setup

Import the basic libraries and define reusable paths to the three raw dataset files. Keeping paths in one place makes the rest of the notebook easier to rerun and adjust.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../dataset")

FEATURES_PATH = DATA_DIR / "elliptic_txs_features.csv"
CLASSES_PATH = DATA_DIR / "elliptic_txs_classes.csv"
EDGES_PATH = DATA_DIR / "elliptic_txs_edgelist.csv"

## Source files

Verify that all expected files are available and record their approximate sizes. This is a quick guard against missing downloads, wrong paths, or accidentally working with incomplete data.

In [2]:
for path in [FEATURES_PATH, CLASSES_PATH, EDGES_PATH]:
    print(path.name, path.exists(), f"{path.stat().st_size / 1024 ** 2:.2f} MB")

elliptic_txs_features.csv True 657.73 MB
elliptic_txs_classes.csv True 3.15 MB
elliptic_txs_edgelist.csv True 4.26 MB


## Load data and assign columns

The features file has no header row, so column names are assigned manually. The first two columns are the transaction identifier and time step; the remaining 165 columns are anonymized numerical features.

In [3]:
feature_cols = ["txId", "time_step"] + [f"f_{i}" for i in range(1, 166)]

features = pd.read_csv(FEATURES_PATH, header=None, names=feature_cols)
classes = pd.read_csv(CLASSES_PATH)
edges = pd.read_csv(EDGES_PATH)

## Dataset dimensions

Summarize the number of rows and columns in each table. At this stage, the features and classes tables should contain the same number of transactions.

In [4]:
summary = pd.DataFrame({
    "dataset": ["features", "classes", "edges"],
    "rows": [len(features), len(classes), len(edges)],
    "columns": [features.shape[1], classes.shape[1], edges.shape[1]],
})

summary

,dataset,rows,columns
0,features,203769,167
1,classes,203769,2
2,edges,234355,2


## Table previews

Preview each raw table to confirm that the columns were read correctly and that the first rows match the expected structure.

### Features preview

The features table should start with `txId` and `time_step`, followed by anonymized numerical feature columns `f_1` through `f_165`.

In [5]:
features.head()

,txId,time_step,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,...,f_156,f_157,f_158,f_159,f_160,f_161,f_162,f_163,f_164,f_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


### Classes preview

The labels table maps each transaction id to one of three values: `1`, `2`, or `unknown`.

In [6]:
classes.head()

,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


### Edges preview

The edgelist represents directed transaction relationships. `txId1` is the source node and `txId2` is the target node.

In [7]:
edges.head()

,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


## Data types

Check whether identifiers, labels, time steps, and numerical features were loaded with sensible types. Unexpected object columns in the features table would usually indicate a parsing problem.

### Feature table types

The expected result is two integer columns (`txId`, `time_step`) and 165 floating-point feature columns.

In [8]:
features.dtypes.value_counts()

float64    165
int64        2
Name: count, dtype: int64

### Label table types

`txId` should be numeric, while `class` may be stored as text because it contains both numeric-looking labels and the `unknown` value.

In [9]:
classes.dtypes

txId     int64
class      str
dtype: object

### Edge table types

Both edge columns should be transaction identifiers stored as integers.

In [10]:
edges.dtypes

txId1    int64
txId2    int64
dtype: object

## Duplicate transaction IDs

Each transaction should appear at most once in the features table and at most once in the classes table. Duplicate ids would make one-to-one merging ambiguous.

In [11]:
audit_duplicates = {
    "features_duplicate_txId": features["txId"].duplicated().sum(),
    "classes_duplicate_txId": classes["txId"].duplicated().sum(),
}

audit_duplicates

{'features_duplicate_txId': np.int64(0), 'classes_duplicate_txId': np.int64(0)}

## Feature-label consistency

Compare transaction ids between the features and classes tables. A clean result means every feature row has exactly one label entry and every label entry has corresponding features.

In [12]:
features_ids = set(features["txId"])
classes_ids = set(classes["txId"])

{
    "features_missing_in_classes": len(features_ids - classes_ids),
    "classes_missing_in_features": len(classes_ids - features_ids),
}

{'features_missing_in_classes': 0, 'classes_missing_in_features': 0}

## Graph endpoint coverage

Check whether all transaction ids appearing in the edgelist also exist in the features table. This determines whether graph metrics can be joined back to the transaction features without losing nodes.

In [13]:
edge_ids = set(edges["txId1"]) | set(edges["txId2"])

{
    "edge_nodes_total": len(edge_ids),
    "edge_nodes_missing_in_features": len(edge_ids - features_ids),
    "features_without_edges": len(features_ids - edge_ids),
}

{'edge_nodes_total': 203769,
 'edge_nodes_missing_in_features': 0,
 'features_without_edges': 0}

## Missing values

Count missing values across all three raw tables. Missing values would require either imputation, filtering, or explicit handling before modeling.

In [14]:
missing_summary = pd.DataFrame({
    "dataset": ["features", "classes", "edges"],
    "missing_values": [
        features.isna().sum().sum(),
        classes.isna().sum().sum(),
        edges.isna().sum().sum(),
    ]
})

missing_summary

,dataset,missing_values
0,features,0
1,classes,0
2,edges,0


## Infinite numeric values

Check for positive or negative infinity in the numeric feature matrix. Infinite values can break scaling, dimensionality reduction, and most machine learning models.

In [15]:
numeric_features = features.drop(columns=["txId"])

np.isinf(numeric_features.to_numpy()).sum()

np.int64(0)

## Class distribution

Inspect the raw label counts. This dataset contains many `unknown` labels, so the supervised classification subset is much smaller than the full transaction graph.

### Label counts

Absolute counts show how many transactions belong to each label category.

In [16]:
classes["class"].value_counts()

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

### Label percentages

Percentages make the class imbalance easier to interpret. In later modeling, this supports using metrics such as precision, recall, F1-score, and PR-AUC instead of accuracy alone.

In [17]:
classes["class"].value_counts(normalize=True).mul(100).round(2)

class
unknown    77.15
2          20.62
1           2.23
Name: proportion, dtype: float64

## Time step audit

The Elliptic dataset has a temporal structure. Auditing `time_step` helps decide whether future model validation should use chronological splits.

### Time step range

Summary statistics confirm the minimum and maximum time step values and give a quick view of the temporal coverage.

In [18]:
features["time_step"].describe()

count    203769.000000
mean         23.843961
std          15.172170
min           1.000000
25%           9.000000
50%          23.000000
75%          38.000000
max          49.000000
Name: time_step, dtype: float64

### Transactions per time step

Counts per time step reveal whether transaction volume is stable over time or concentrated in specific periods.

In [19]:
features["time_step"].value_counts().sort_index()

time_step
1     7880
2     4544
3     6621
4     5693
5     6803
6     4328
7     6048
8     4457
9     4996
10    6727
11    4296
12    2047
13    4528
14    2022
15    3639
16    2975
17    3385
18    1976
19    3506
20    4291
21    3537
22    5894
23    4165
24    4592
25    2314
26    2523
27    1089
28    1653
29    4275
30    2483
31    2816
32    4525
33    3151
34    2486
35    5507
36    6393
37    3306
38    2891
39    2760
40    4481
41    5342
42    7140
43    5063
44    4975
45    5598
46    3519
47    5121
48    2954
49    2454
Name: count, dtype: int64

## Merge validation

The `features` and `classes` tables should join one-to-one on `txId`. This merged dataframe is the base table for the next analysis notebooks.

In [20]:
df = features.merge(classes, on="txId", how="left", validate="one_to_one")

df.shape

(203769, 168)

In [21]:
df["class"].value_counts(dropna=False)

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

In [22]:
df.head()

,txId,time_step,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,...,f_157,f_158,f_159,f_160,f_161,f_162,f_163,f_164,f_165,class
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,unknown
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,unknown
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,2
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117,unknown


## Known labels subset

For standard supervised binary classification, only known labels should be used. The `unknown` rows can be kept for later prediction or semi-supervised graph experiments.

In [23]:
known_df = df[df["class"].astype(str).isin(["1", "2"])].copy()
known_df["target"] = known_df["class"].astype(str).map({"1": 1, "2": 0})

pd.DataFrame({
    "subset": ["all_transactions", "known_labels", "unknown_labels"],
    "rows": [len(df), len(known_df), (df["class"].astype(str) == "unknown").sum()],
    "share_pct": [
        100.0,
        len(known_df) / len(df) * 100,
        (df["class"].astype(str) == "unknown").mean() * 100,
    ],
}).round({"share_pct": 2})

,subset,rows,share_pct
0,all_transactions,203769,100.00
1,known_labels,46564,22.85
2,unknown_labels,157205,77.15


In [24]:
known_df["target"].value_counts().rename(index={1: "illicit", 0: "licit"})

target
licit      42019
illicit     4545
Name: count, dtype: int64

## Audit report

Save a compact JSON summary that can be reused in later notebooks or referenced in the project report.

In [25]:
import json

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

class_counts = classes["class"].value_counts().to_dict()

audit_report = {
    "n_transactions": int(len(features)),
    "n_edges": int(len(edges)),
    "n_feature_columns_total": int(features.shape[1]),
    "n_engineered_feature_columns": int(features.shape[1] - 2),
    "time_step_min": int(features["time_step"].min()),
    "time_step_max": int(features["time_step"].max()),
    "class_counts": {str(k): int(v) for k, v in class_counts.items()},
    "known_label_rows": int(len(known_df)),
    "unknown_label_rows": int((df["class"].astype(str) == "unknown").sum()),
    "features_duplicate_txId": int(features["txId"].duplicated().sum()),
    "classes_duplicate_txId": int(classes["txId"].duplicated().sum()),
    "features_missing_in_classes": int(len(features_ids - classes_ids)),
    "classes_missing_in_features": int(len(classes_ids - features_ids)),
    "edge_nodes_total": int(len(edge_ids)),
    "edge_nodes_missing_in_features": int(len(edge_ids - features_ids)),
    "features_without_edges": int(len(features_ids - edge_ids)),
    "missing_values_total": int(
        features.isna().sum().sum()
        + classes.isna().sum().sum()
        + edges.isna().sum().sum()
    ),
    "infinite_numeric_values": int(np.isinf(numeric_features.to_numpy()).sum()),
}

report_path = OUTPUT_DIR / "00_dataset_audit_report.json"
with report_path.open("w") as f:
    json.dump(audit_report, f, indent=2)

audit_report

{'n_transactions': 203769,
 'n_edges': 234355,
 'n_feature_columns_total': 167,
 'n_engineered_feature_columns': 165,
 'time_step_min': 1,
 'time_step_max': 49,
 'class_counts': {'unknown': 157205, '2': 42019, '1': 4545},
 'known_label_rows': 46564,
 'unknown_label_rows': 157205,
 'features_duplicate_txId': 0,
 'classes_duplicate_txId': 0,
 'features_missing_in_classes': 0,
 'classes_missing_in_features': 0,
 'edge_nodes_total': 203769,
 'edge_nodes_missing_in_features': 0,
 'features_without_edges': 0,
 'missing_values_total': 0,
 'infinite_numeric_values': 0}

In [26]:
report_path

PosixPath('../outputs/00_dataset_audit_report.json')

## Conclusions

- The dataset contains 203,769 transactions and 234,355 directed graph edges.
- The features file contains `txId`, `time_step`, and 165 anonymized numerical features.
- The labels file contains three label values: `1`, `2`, and `unknown`.
- For supervised binary classification, only labels `1` and `2` should be used; `unknown` rows should not be treated as a third supervised class.
- The dataset spans 49 time steps, so later modeling should prefer a chronological split over a purely random split.
- All edge endpoints are present in the features table, so graph-based analysis can be joined safely with transaction features.